# 05 · Modelo diario: ¿qué explica que un día haya huaico?

Pregunta: con la información disponible hasta *ayer* (suelo, lluvia previa, caudal, temperatura del mar, calor y humedad aguas arriba, estado de El Niño), ¿se puede decir mejor que con el calendario si hoy es un día con huaico catalogado en Chosica, Chaclacayo, Santa Eulalia o Ricardo Palma? ¿Y cuánto suma la lluvia del propio día?

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Mide cuánto del catálogo se explica con el tiempo atmosférico, con las fechas y los huecos del catálogo tal como están. Ante una emergencia llama al 115 (INDECI).

Siete teorías, escritas antes de ver los resultados: T1 solo la fecha; T2 lluvia del día; T3 antecedentes (todo desplazado un día, sin la lluvia de hoy); T4 lluvia más antecedentes; T5 más el ICEN causal (T5b: más la marca de temporada Niño, que solo se sabe a posteriori); T6 no lineal (árboles muy pequeños); T7 dos etapas. La regularización se elige **dentro** de cada conjunto de entrenamiento y la prueba deja fuera temporadas completas.

Requisitos: `uv run descargar-datos --solo open-meteo-predictores noaa-oisst open-meteo-caudal igp-icen open-meteo-horario noaa-enso` (y `--solo era5-aws` para los puntos que la API gratuita no entregó; capítulo 13). El notebook usa lo que ya esté descargado y dice qué datos usó. Las cifras del capítulo [`docs/05-modelo-diario.md`](../docs/05-modelo-diario.md) salieron de una corrida en la PC con GPU; aquí se puede repetir con menos mundos nulos.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from illapa_research import evaluacion_diaria as E
from illapa_research import modelo_diario as M

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
RAIZ = Path("..") if Path("../datos").exists() else Path(".")
N_JOBS = -1

D = M.cargar_datos(RAIZ)
X, y, ts, mes = M.matrices(D)
inv = {k: v for k, v in D.inventario.items() if k != "mask"}
print(inv)
print(f"{len(y)} días de temporada (dic-abr, 1984 en adelante), {int(y.sum())} con huaico ({y.mean():.1%}), {len(np.unique(ts))} temporadas")

## 1. Las siete teorías, dejando fuera una temporada por vez

`aciertos_con_5pc_alarmas` = qué parte de los días con huaico cae entre el 5 % de días con mayor probabilidad. `brier_skill` y `logloss_ganancia` se miden contra la climatología mensual (T1).

In [ ]:
res = M.correr_teorias(D, "temporada", n_jobs=N_JOBS)
folds = [f for f in E.pliegues(ts, "temporada") if len(f)]
tabla = M.tabla_metricas(res, y)
tabla.round(3)

**Intervalos del 90 %** remuestreando temporadas completas: es la incertidumbre que importa, porque los días con huaico vienen en racimos (23 días en 2017).

In [ ]:
iv = {k: M.intervalos_por_temporada(y.to_numpy(), res[k].p, res["fecha"].p, ts, n=300)
      for k in ["lluvia_dia", "antecedentes", "lluvia+antecedentes", "+nino", "no_lineal"]}
pd.concat({k: v.loc[["ap", "auc", "brier_skill", "aciertos_con_5pc_alarmas"]] for k, v in iv.items()}).round(3)

## 2. ¿Es mejor que el azar? Prueba nula con miles de mundos

Se desplazan cíclicamente los días con huaico dentro de cada temporada (mismo número de eventos y mismo racimado, sin relación con el tiempo) y se repite la validación dejando una temporada fuera, con una regularización fija (C = 0,03, la que más se elige). Se usan 2 000 mundos con GPU (`illapa_research.gpu`); sin GPU, 200 para que sea rápido. `p` es la fracción de mundos nulos que alcanza el AP real (más uno).

In [ ]:
from illapa_research import gpu

N_NULAS = 2000 if gpu.dispositivo() != "cpu" else 200
M.tabla_nulas(X, y, ts, n_nulas=N_NULAS).round(4)

## 3. Validaciones alternativas: episodios Niño y bloques de cinco temporadas

In [ ]:
resE = M.correr_teorias(D, "episodio", n_jobs=N_JOBS, con_oraculo=False)
folds_b = M.pliegues_bloques(ts, 5)
resB = {k: M.validar_anidado(X[k], y, ts, folds_b, n_jobs=N_JOBS)
        for k in ["lluvia_dia", "antecedentes", "lluvia+antecedentes", "+nino"]}
resB["fecha"] = M.Resultado(E.p_climatologia(y, mes, folds_b), [], None, [])
pd.concat({"episodio Niño fuera": M.tabla_metricas(resE, y)[["ap", "auc", "brier_skill", "aciertos_con_5pc_alarmas"]],
           "bloque de 5 temporadas fuera": M.tabla_metricas(resB, y)[["ap", "auc", "brier_skill", "aciertos_con_5pc_alarmas"]]}).round(3)

## 4. ¿Qué familia de variables aporta? Ablación

Al modelo T5 se le quita una familia por vez (todas desplazadas un día). Una caída grande del `ap` quiere decir que esa familia aporta algo que las otras no tienen.

In [ ]:
yv = y.to_numpy()
pref = E.p_climatologia(y, mes, folds)
filas = {"completo (T5)": M.metricas_completas(yv, res["+nino"].p, pref)}
for fam in ["mar", "caudal", "calor_altura", "humedad_y_circulacion", "suelo", "lluvia_previa", "nino_icen"]:
    filas[f"sin {fam}"] = M.metricas_completas(yv, M.ablacion(D, fam, n_jobs=N_JOBS).p, pref)
pd.DataFrame(filas).T[["ap", "auc", "brier_skill", "aciertos_con_5pc_alarmas"]].round(3)

## 5. Niño y no Niño, calibración y estabilidad

¿El modelo funciona igual dentro de temporadas Niño que fuera? `observado_sobre_predicho` mayor que 1 = el modelo se queda corto.

In [ ]:
pd.concat({k: M.por_regimen(y, res[k], ts, D.temporadas) for k in ["lluvia_dia", "lluvia+antecedentes", "+nino"]}).round(3)

In [ ]:
M.calibracion(yv, res["+nino"].p).round(4)

In [ ]:
est = M.estabilidad_coeficientes(res["lluvia+antecedentes"])
print(f"{int(est.estable.sum())} de {len(est)} coeficientes activos y con el mismo signo en al menos 80 % de los pliegues")
from collections import Counter

print("regularización elegida por pliegue (T5):", Counter(res["+nino"].configs).most_common())
est.head(12).round(3)

Los coeficientes estables **no se interpretan como causas**: con 91 columnas muy correlacionadas el signo de una puede compensar el de otra. Sirven para ver si el modelo es consistente entre pliegues.

## 6. Robustez

Misma validación, otras etiquetas y ventanas: día del evento y el anterior (por si el catálogo fecha un día tarde los eventos nocturnos), desde 2009 (registro más consistente) y sin los tres mayores racimos.

In [ ]:
rob = M.robustez(RAIZ, n_jobs=N_JOBS)
rob.pivot(index="teoria", columns="escenario", values="ap").loc[["fecha", "lluvia_dia", "antecedentes", "lluvia+antecedentes", "+nino"]].round(3)

In [ ]:
rob.pivot(index="teoria", columns="escenario", values="aciertos_con_5pc_alarmas").loc[["lluvia_dia", "antecedentes", "lluvia+antecedentes", "+nino"]].round(3)

## 6b. Etiqueta del día anterior y definición alterna de Niño

Todos los pasos de arriba repetidos con (a) la etiqueta «el día del evento y el anterior» y (b) la definición alterna de temporada Niño (anomalía Niño 1+2 en lugar del ICEN oficial), que cambia los episodios, la separación Niño / no Niño y el modelo oráculo. Cada corrida tarda unos minutos.

In [ ]:
variantes = {
    "día anterior": M.informe(RAIZ, dias_antes=1, n_nulas=N_NULAS, n_jobs=N_JOBS, con_ablacion=False),
    "Niño alterno (Niño 1+2)": M.informe(RAIZ, nino="n12", n_nulas=N_NULAS, n_jobs=N_JOBS, con_ablacion=False),
}
pd.concat({k: v["teorias"][["ap", "auc", "brier_skill", "aciertos_con_5pc_alarmas"]] for k, v in variantes.items()}).round(3)

In [ ]:
pd.concat({k: v["nulas"][["ap_real", "ap_nula_p95", "ap_nula_max", "p"]] for k, v in variantes.items()}).round(4)

In [ ]:
pd.concat({k: pd.concat(v["regimen"]) for k, v in variantes.items()}).round(3)

## 7. ¿Cuánto es sobreajuste?

Con ~93 columnas y ~118 días con huaico, el ajuste dentro de la muestra es optimista. La validación dejando temporadas fuera es la cifra honesta; aquí se ve la brecha y cuánto se conserva con un modelo de 11 variables elegidas por conocimiento previo (análisis posterior, no pre-registrado).

In [ ]:
print(M.sobreajuste(X["+nino"], y, ts, res["+nino"]))
sel = ["chosica_h_lluvia_1d", "chosica_h_lluvia_3d", "chosica_h_pico", "mat_lluvia_1d", "chosica_suelo_prev",
       "chosica_h_lluvia_7d_previa_prev", "sst_nino12_anom_30d_prev", "sea_temp_anom_prev",
       "sea_radiacion_anom_prev", "caudal_rimac-chosica_max_7d_prev", "icen_mes_previo"]
sel = [c for c in sel if c in X["+nino"].columns]
par = M.validar_anidado(X["+nino"][sel], y, ts, folds, n_jobs=N_JOBS)
pd.Series(M.metricas_completas(yv, par.p, pref)).round(3)

## Qué NO se puede concluir

- Esto no es una alerta: aun con la mejor teoría, cuatro de cada cinco alarmas del 5 % de días más probables no coinciden con un huaico catalogado, y casi la mitad de los días con huaico queda sin alarma.
- El catálogo es de prensa e instituciones; un día sin huaico catalogado no es un día sin huaico. La lluvia y el calor vienen de ERA5 (reanálisis de 25 km), no de pluviómetros en las cabeceras.
- Los resultados dependen de 1983 a 2026 con pocos racimos grandes (2015, 2017, 2019, 2023). Hay que releerlos cuando lleguen los demás puntos de ERA5 (Chosica, costa de Lima, selva) y las cajas de temperatura del mar que faltan.